# 🎨 Tool Calling Tutorial: Synthetic Data with Data Designer

#### 📚 What you'll learn

This notebook demonstrates how to use NMP Data Designer to generate synthetic tool calling training data:

- How to define tool schemas using Pydantic models and generate structured JSON
- How to use sampler columns to steer diversity across tool domains
- How to judge quality and transform output to fine-tuning format
- How to iterate with preview, then scale up to a full dataset

If this is your first time with this tutorial series, start with [Notebook 1: Data Preparation](./1_data_preparation.ipynb).


### 📦 Imports

- `nemo_microservices` provides the NMP platform client.
- `NeMoDataDesignerClient` is the interface for data generation on NMP.
- `nemo_microservices.data_designer.config` provides column configs, samplers, and processors.


### ⚡ Setup

Install dependencies and configure your environment.


In [ ]:
%%capture
!pip install -r requirements.txt

In [ ]:
from nemo_microservices import NeMoMicroservices
from nemo_microservices.data_designer.client import NeMoDataDesignerClient
import nemo_microservices.data_designer.config as dd
from pydantic import BaseModel, Field
from typing import List, Dict, Any

from config import NEMO_URL, NIM_URL, WORKSPACE, DD_TRAINING_FILESET

### ⚙️ Initialize the NeMo Microservices client

- `NeMoMicroservices` connects to the NMP platform.
- `NeMoDataDesignerClient` wraps the client for data generation workflows.


In [ ]:
client = NeMoMicroservices(
    base_url=NEMO_URL,
    inference_base_url=NIM_URL,
    workspace=WORKSPACE,
)

dd_client = NeMoDataDesignerClient(client=client)

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model for use during generation.
- The model alias is referenced in column configs below.
- We use Nemotron Super for high-quality generation.


In [ ]:
MODEL_PROVIDER = "default/build-nvidia"
MODEL_ID = "nvidia/nemotron-super-llama-3.3-49b"
MODEL_ALIAS = "generator"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=0.9,
            top_p=0.95,
            max_tokens=1024,
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The config builder provides an intuitive interface for defining your dataset schema.
- The list of model configs is provided at initialization.


In [ ]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

## 🎲 Steer diversity with sampler columns

- Sampler columns ensure generated data covers a wide range of tool domains.
- This is key for training a model that generalizes well across use cases.


In [ ]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="domain",
        sampler_type="category",
        params={"values": [
            "weather",
            "calendar",
            "web_search",
            "math",
            "file_management",
            "database",
            "email",
            "navigation",
            "e_commerce",
            "social_media",
        ]},
    )
)

## 🧑‍🎨 Define tool calling schemas

- We use Pydantic models to define the exact structure of our tool definitions and tool calls.
- Data Designer enforces these schemas during generation.

> 💡 **Why Pydantic?**
>
> - Pydantic models provide IDE support and type validation.
> - They integrate directly with Data Designer's `LLMStructuredColumnConfig`.


In [ ]:
class ParameterDef(BaseModel):
    """A single parameter in a function definition."""
    name: str = Field(description="Parameter name")
    type: str = Field(description="JSON schema type (string, integer, number, boolean, array, object)")
    description: str = Field(description="What this parameter does")


class FunctionDef(BaseModel):
    """A tool/function definition."""
    name: str = Field(description="Function name in snake_case")
    description: str = Field(description="What this function does")
    parameters: List[ParameterDef] = Field(description="List of function parameters")


class ToolDefinitions(BaseModel):
    """A set of available tools."""
    tools: List[FunctionDef] = Field(description="2-4 tool/function definitions")


class ToolCallArg(BaseModel):
    """A single tool call with arguments."""
    name: str = Field(description="Name of the function to call")
    arguments: Dict[str, Any] = Field(description="Arguments to pass to the function")


class ExpectedToolCalls(BaseModel):
    """The expected tool calls for a given query."""
    tool_calls: List[ToolCallArg] = Field(description="One or more tool calls")

## 🦜 Generate tool definitions

- `LLMStructuredColumnConfig` generates JSON conforming to our Pydantic schema.
- The prompt references `{{ domain }}` from our sampler column.


In [ ]:
config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="tools",
        prompt=(
            "Generate 2-4 realistic API function definitions for the '{{ domain }}' domain. "
            "Each function should have a descriptive snake_case name, clear description, "
            "and well-typed parameters. Make them diverse — include functions with "
            "different parameter counts and types."
        ),
        output_format=ToolDefinitions,
        model_alias=MODEL_ALIAS,
    )
)

## 🦜 Generate user queries

- `LLMTextColumnConfig` generates natural language user queries.
- The prompt references `{{ tools }}` to ensure queries match available tools.


In [ ]:
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="user_query",
        prompt=(
            "Given these available tools:\n{{ tools }}\n\n"
            "Generate a natural, realistic user query that would require calling "
            "one or more of these tools. The query should be something a real user "
            "would type. Vary complexity — sometimes simple, sometimes requiring "
            "multiple tool calls. Respond with only the query, no other text."
        ),
        model_alias=MODEL_ALIAS,
    )
)

## 🦜 Generate expected tool calls

- Given a user query and available tools, generate the correct tool call(s).
- This is the "ground truth" that the fine-tuned model should learn to produce.


In [ ]:
config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="expected_tool_calls",
        prompt=(
            "Given this user query: '{{ user_query }}'\n"
            "And these available tools: {{ tools }}\n\n"
            "Determine exactly which tool(s) should be called and with what arguments. "
            "Be precise with argument values — they should directly address the user's request."
        ),
        output_format=ExpectedToolCalls,
        model_alias=MODEL_ALIAS,
    )
)

## ⚖️ Judge quality

- `LLMJudgeColumnConfig` scores each example across multiple quality dimensions.
- Low-quality examples can be filtered out before fine-tuning.


In [ ]:
config_builder.add_column(
    dd.LLMJudgeColumnConfig(
        name="quality_score",
        prompt=(
            "Evaluate this tool calling example:\n"
            "User query: {{ user_query }}\n"
            "Available tools: {{ tools }}\n"
            "Tool calls made: {{ expected_tool_calls }}"
        ),
        model_alias=MODEL_ALIAS,
        scores=[
            {
                "name": "correctness",
                "description": "Are the correct tools called with valid arguments?",
                "options": {
                    "1": "Wrong tool or invalid arguments",
                    "2": "Partially correct",
                    "3": "Fully correct",
                },
            },
            {
                "name": "naturalness",
                "description": "Is the user query realistic and natural?",
                "options": {
                    "1": "Artificial or contrived",
                    "2": "Acceptable",
                    "3": "Very natural",
                },
            },
        ],
    )
)

## 🔄 Transform to fine-tuning format

- `SchemaTransformProcessorConfig` reshapes columns into the OpenAI messages+tools format.
- This is the format NeMo Customizer expects.


In [ ]:
config_builder.add_processor(
    dd.SchemaTransformProcessorConfig(
        name="to_openai_format",
        template={
            "messages": [
                {"role": "user", "content": "{{ user_query }}"},
                {"role": "assistant", "tool_calls": "{{ expected_tool_calls }}"},
            ],
            "tools": "{{ tools }}",
        },
    )
)

### 🔁 Iteration is key – preview the dataset!

1. Use the `preview` method to generate a small sample quickly.
2. Inspect the results for quality and format issues.
3. Adjust column configurations, prompts, or parameters as needed.
4. Re-run the preview until satisfied.


In [ ]:
preview = dd_client.preview(config_builder, num_records=2)

In [ ]:
preview.display_sample_record()

In [ ]:
preview.dataset

### 📊 Analyze the generated data

- Data Designer automatically profiles the generated data.


In [ ]:
preview.analysis.to_report()

### 🆙 Scale up!

- Happy with your preview data?
- Use the `create` method to generate the full training dataset.


In [ ]:
job = dd_client.create(config_builder, num_records=5000, wait_until_done=True)

In [ ]:
dataset = job.load_dataset()
dataset.head()

## 📤 Upload to NMP Filesets

- Upload the generated dataset so the Customizer can use it for fine-tuning.


In [ ]:
import os

output_path = os.path.join(os.getcwd(), "data", "dd_training.jsonl")
dataset.to_json(output_path, orient="records", lines=True)

try:
    client.filesets.create(name=DD_TRAINING_FILESET, description="Data Designer generated tool calling data")
    print(f"Created fileset: {DD_TRAINING_FILESET}")
except Exception as e:
    if "409" in str(e):
        print(f"Fileset {DD_TRAINING_FILESET} already exists")
    else:
        raise

with open(output_path, "rb") as f:
    client.filesets.upload_file(name=DD_TRAINING_FILESET, path="training/training.jsonl", body=f.read())

print(f"Uploaded DD-generated data to fileset: {DD_TRAINING_FILESET}")

## ⏭️ Next Steps

Your synthetic tool calling dataset is ready! In the next notebook, we'll fine-tune Nemotron Nano using this data.

- [3. Fine-Tuning and Inference](./3_finetuning_and_inference.ipynb)
